# CS 521 HW3, Problem 2: SynCode

In [ ]:
!pip install -q --ignore-requires-python syncode==0.4.16

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [ ]:
import operator

from syncode import Syncode

GRAMMAR = r"""
start: " " number " " OPERATOR " " number "\n"
number: INT "." INT
INT: /[0-9]+/
OPERATOR: "+" | "-" | "*" | "/"
"""

FEW_SHOT = """Extract the two operands and the operator (+, -, *, /) from the query.

Query: What is 15.5 plus 4.25?
Extraction: 15.5 + 4.25

Query: What is 100. minus 37.8?
Extraction: 100.0 - 37.8

Query: What is 6.0 multiplied by 7.?
Extraction: 6.0 * 7.0

Query: What is 81.9 divided by 9.0?
Extraction: 81.9 / 9.0

"""

syn_llm = Syncode(model="microsoft/phi-2", mode="grammar_strict", grammar=GRAMMAR, parse_output_only=True, max_new_tokens=30)

[2026-10-27 01:45:15,466-root] - Loading model microsoft/phi-2 with device:cuda, device_map:auto, torch_dtype:torch.bfloat16
[2026-10-27 01:45:15,622-accelerate.utils.modeling] - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

[2026-10-27 01:45:38,332-syncode.mask_store.mask_store] - Using cache: True and fsm path cache/mask_stores/CodeGenTokenizerFast/grammar_strict_7352800764_50257.pkl exist: False
[2026-10-27 01:45:38,333-syncode.mask_store.mask_store] - Creating mask store for CodeGenTokenizerFast and custom, may take more than 10 minutes. Caching at /content/cache/mask_stores/CodeGenTokenizerFast/grammar_strict_7352800764_50257.pkl.
[2026-10-27 01:45:38,842-syncode.mask_store.fsm_set] - 5 FSMs with 14 states initialized in 0.00 seconds
[2026-10-27 01:45:38,843-syncode.mask_store.mask_store] - Ignore whitespace is False
[2026-10-27 01:45:38,846-syncode.mask_store.mask_store] - Number of 2 length terminal sequences reduced from 25 to 7


100%|██████████| 14/14 [00:03<00:00,  3.86it/s]

[2026-10-27 01:45:42,492-syncode.mask_store.mask_store] - Time taken to create mask store: 4.15 seconds


In [7]:
OPS = {"+": operator.add, "-": operator.sub, "*": operator.mul, "/": operator.truediv}


def run_syncode(textual_query):
    output = syn_llm.infer(FEW_SHOT + f"Query: {textual_query}\nExtraction:")[0]
    a, op, b = output.split()
    operands = [float(a), float(b)]
    return operands, op, OPS[op](*operands)

In [8]:
for query in ["What is 327. multiplied by 11.0?", "What is 45.1 plus 23.54?", "What is 120.4 divided by 4.0?"]:
    print(query, run_syncode(query))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


What is 327. multiplied by 11.0? ([327.0, 11.0], '*', 3597.0)
What is 45.1 plus 23.54? ([45.1, 23.54], '+', 68.64)
What is 120.4 divided by 4.0? ([120.4, 4.0], '/', 30.1)
